# Bước 1: Tiền xử lý & RFM (Online Retail II)

## Quy ước tính RFM — **đã chốt**

| Chỉ số | Công thức | Ghi chú |
|--------|-----------|---------|
| **R — Recency** | $R_i = \bigl(t_{\mathrm{ref}} - \max(\text{InvoiceDate của khách } i)\bigr)$ tính theo **ngày** | $t_{\mathrm{ref}} = \max(\text{InvoiceDate})$ trên **toàn tập sau khi làm sạch**. **R càng nhỏ** → khách mua **gần đây** hơn. |
| **F — Frequency** | $F_i =$ số giá trị **Invoice** **khác nhau** của khách $i$ | Đếm **số đợt mua / số hóa đơn**, không phải số dòng sản phẩm. |
| **M — Monetary** | $M_i = \sum (\text{Quantity} \times \text{Price})$ trên các dòng hợp lệ của khách $i$ | Tổng giá trị dòng `Quantity × Price`. |

## Làm sạch — **đã chốt**

1. Loại dòng thiếu `Customer ID`.
2. Loại giao dịch **hoàn / hủy**: cột `Invoice` (chuỗi) **bắt đầu bằng `C`** (quy ước bộ Online Retail).
3. Loại dòng có `Quantity ≤ 0` hoặc `Price ≤ 0`.
4. Chuyển `InvoiceDate` sang `datetime`.

*Tùy chọn trong code:* lọc một `Country` (ví dụ chỉ UK). Mặc định **không lọc** — giữ mọi quốc gia sau bước trên.

## Chuẩn hóa cho phân cụm — **đã chốt**

- $F$ và $M$ thường **lệch phải** → dùng $F' = \log(1+F)$, $M' = \log(1+M)$ (log1p).
- $R$ **không** log (thang ngày đã diễn giải trực tiếp).
- Trên $(R, F', M')$ áp dụng **StandardScaler** (z-score theo từng cột): trung bình 0, phương sai 1 — để khoảng cách Euclidean không bị một trục (thường là M) áp đảo.

**Output:** thư mục `data/` — file `rfm_customers.csv` gồm R, F, M gốc, `F_log1p`, `M_log1p`, và `R_z`, `F_z`, `M_z`.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Gốc project = thư mục có requirements.txt (notebook có thể nằm trong notebooks/...)
def _project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "requirements.txt").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()


ROOT = _project_root()
DATA_PATH = ROOT / "online_retail_II.csv"
OUT_DIR = ROOT / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Tùy chọn: chỉ phân tích một quốc gia (ví dụ "United Kingdom"). None = giữ tất cả.
FILTER_COUNTRY = None  # hoặc: "United Kingdom"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Đọc toàn bộ CSV (~1M+ dòng)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df.shape)
print(df.head(3))

Raw shape: (1067371, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  


In [3]:
def clean_online_retail(
    raw: pd.DataFrame,
    *,
    filter_country: Optional[str] = None,
) -> pd.DataFrame:
    """Áp dụng các quy tắc làm sạch đã chốt (xem ô Markdown đầu)."""
    d = raw.copy()

    # Chuẩn hóa tên cột nếu có khoảng trắng thừa
    d.columns = [c.strip() for c in d.columns]

    # Parse ngày
    d["InvoiceDate"] = pd.to_datetime(d["InvoiceDate"], errors="coerce")

    # 1) Bỏ thiếu Customer ID
    d = d.dropna(subset=["Customer ID"])

    # 2) Bỏ hóa đơn hủy (prefix C)
    inv = d["Invoice"].astype(str)
    d = d[~inv.str.startswith("C")]

    # 3) Chỉ giữ giao dịch có Quantity > 0 và Price > 0
    d = d[(d["Quantity"] > 0) & (d["Price"] > 0)]

    # Bỏ dòng ngày không parse được
    d = d.dropna(subset=["InvoiceDate"])

    if filter_country is not None:
        d = d[d["Country"] == filter_country]

    # Dòng tổng tiền
    d["LineTotal"] = d["Quantity"].astype(float) * d["Price"].astype(float)

    return d


df_clean = clean_online_retail(df, filter_country=FILTER_COUNTRY)
print("After clean shape:", df_clean.shape)
print("Customers:", df_clean["Customer ID"].nunique())
print("Date range:", df_clean["InvoiceDate"].min(), "→", df_clean["InvoiceDate"].max())

After clean shape: (805549, 9)
Customers: 5878
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


In [4]:
# Ngày tham chiếu: mốc thời gian mới nhất trong tập đã làm sạch
t_ref = df_clean["InvoiceDate"].max()

# Gom theo khách hàng — công thức RFM đã chốt
rfm = (
    df_clean.groupby("Customer ID", as_index=False)
    .agg(
        R=("InvoiceDate", lambda s: (t_ref - s.max()).days),
        F=("Invoice", "nunique"),
        M=("LineTotal", "sum"),
    )
)

rfm = rfm.rename(columns={"Customer ID": "customer_id"})
# Đảm bảo kiểu số nguyên cho R, F
rfm["R"] = rfm["R"].astype(int)
rfm["F"] = rfm["F"].astype(int)

print(rfm.describe())
rfm.head(10)

        customer_id            R            F              M
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   200.331916     6.289384    3018.616737
std     1715.572666   209.338707    13.009406   14737.731040
min    12346.000000     0.000000     1.000000       2.950000
25%    13833.250000    25.000000     1.000000     348.762500
50%    15314.500000    95.000000     3.000000     898.915000
75%    16797.750000   379.000000     7.000000    2307.090000
max    18287.000000   738.000000   398.000000  608821.650000


,customer_id,R,F,M
0,12346.0,325,12,77556.46
1,12347.0,1,8,5633.32
2,12348.0,74,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,309,1,334.40
5,12351.0,374,1,300.93
6,12352.0,35,10,2849.84
7,12353.0,203,2,406.76
8,12354.0,231,1,1079.40
9,12355.0,213,2,947.61


In [5]:
# Log1p cho F, M — đã chốt
rfm["F_log1p"] = np.log1p(rfm["F"].astype(float))
rfm["M_log1p"] = np.log1p(rfm["M"].astype(float))

# Z-score trên (R, F_log1p, M_log1p) — đã chốt
feat_cols = ["R", "F_log1p", "M_log1p"]
scaler = StandardScaler()
Z = scaler.fit_transform(rfm[feat_cols])
rfm["R_z"] = Z[:, 0]
rfm["F_z"] = Z[:, 1]
rfm["M_z"] = Z[:, 2]

out_path = OUT_DIR / "rfm_customers.csv"
rfm.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())
print("Rows:", len(rfm))
rfm.head()

Saved: /Users/kotori/GMS_AFKMC2/data/rfm_customers.csv
Rows: 5878


,customer_id,R,F,M,F_log1p,M_log1p,R_z,F_z,M_z
0,12346.0,325,12,77556.46,2.564949,11.258774,0.595584,1.254496,3.186625
1,12347.0,1,8,5633.32,2.197225,8.636632,-0.952279,0.800166,1.297127
2,12348.0,74,5,2019.40,1.791759,7.611051,-0.603532,0.299207,0.558100
3,12349.0,18,4,4428.69,1.609438,8.396085,-0.871064,0.073946,1.123790
4,12350.0,309,1,334.40,0.693147,5.815324,0.519146,-1.058146,-0.735888


In [6]:
# Kiểm tra nhanh
assert rfm["R"].min() >= 0
assert rfm[["F", "M"]].min().min() > 0
print("Số khách hàng sau RFM:", len(rfm))
print("Z-score — mean ~ 0, std ~ 1:", rfm[["R_z", "F_z", "M_z"]].mean().round(4).to_dict())
print("Z-score std:", rfm[["R_z", "F_z", "M_z"]].std().round(4).to_dict())

Số khách hàng sau RFM: 5878
Z-score — mean ~ 0, std ~ 1: {'R_z': -0.0, 'F_z': 0.0, 'M_z': -0.0}
Z-score std: {'R_z': 1.0001, 'F_z': 1.0001, 'M_z': 1.0001}
